# Lab 5: tools and an agent loop

Give a model four tools and a loop, and watch every step go by: the tool definitions, the
checks on each request, the approval in front of an order, and the step limit. This notebook
follows [Lab 5 in the book](https://handbook.aggarwal.dev/lab-agent.html): read each step there, then run it here.
The code is in `labs/05-agent/agent.py`.

## Set up

Run the next cell first. It finds the book's code (in Colab it downloads it), installs what
the labs need and picks who answers:

- `fake` needs no key and costs nothing. Its replies are scripted, so they show how the
  code works, not what a model would say.
- `anthropic` uses Claude with your own API key. In Colab, add `ANTHROPIC_API_KEY` in the
  Secrets panel (the key icon on the left) and let this notebook read it. Elsewhere the cell
  asks for the key. It is never written into the notebook.
- `ollama` runs a small open model, free. In Colab it installs and starts Ollama for you:
  switch to a GPU first (Runtime, then Change runtime type, then T4 GPU), and expect the
  first run to download about 2 GB.

To change the backend later, change it here and run this cell again. Lab 0 in the book
explains the choices: https://handbook.aggarwal.dev/lab-setup.html

In [ ]:
# @title Set up this lab
BACKEND = "fake"  # @param ["fake", "anthropic", "ollama"]
MODEL = ""  # @param {type:"string"}

import os, shutil, subprocess, sys, time, urllib.request
from pathlib import Path

REPO = "https://github.com/aggarwaldev/build-ai-handbook.git"


def find_root():
    here = Path.cwd().resolve()
    for folder in (here, *here.parents):
        if (folder / "labs" / "common" / "llm.py").is_file():
            return folder
    return None


root = find_root()
if root is None:
    if not Path("build-ai-handbook").is_dir():
        subprocess.run(["git", "clone", "--quiet", "--depth", "1", REPO], check=True)
    root = Path("build-ai-handbook").resolve()
os.chdir(root)


def run_command(*cmd):
    """Run a command from the repository root, show its output in this cell, and stop if it fails."""
    done = subprocess.run([sys.executable if part == "python" else part for part in cmd],
                          capture_output=True, text=True)
    print(done.stdout, end="")
    print(done.stderr, end="", file=sys.stderr)
    if done.returncode:
        raise RuntimeError("this command failed: " + " ".join(cmd))


if str(root / "labs") not in sys.path:
    sys.path.insert(0, str(root / "labs"))

try:
    import anthropic, numpy  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-r", "labs/requirements.txt"], check=True)

os.environ["LLM_BACKEND"] = BACKEND
if MODEL.strip():
    os.environ["LLM_MODEL"] = MODEL.strip()
else:
    os.environ.pop("LLM_MODEL", None)
IN_COLAB = "google.colab" in sys.modules

if BACKEND == "anthropic" and not os.environ.get("ANTHROPIC_API_KEY", "").strip():
    if IN_COLAB:
        from google.colab import userdata
        try:
            os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
        except Exception as err:
            raise RuntimeError("Add ANTHROPIC_API_KEY in the Secrets panel (the key icon on the left) "
                               "and allow this notebook to read it, then run this cell again.") from err
    else:
        from getpass import getpass
        os.environ["ANTHROPIC_API_KEY"] = getpass("Anthropic API key (not saved in the notebook): ")


def ollama_running():
    host = os.environ.get("OLLAMA_HOST", "http://localhost:11434").rstrip("/")
    try:
        urllib.request.urlopen(host + "/api/tags", timeout=2)
        return True
    except OSError:
        return False


if BACKEND == "ollama":
    if not ollama_running():
        if not IN_COLAB:
            raise RuntimeError("Start Ollama on this machine (ollama serve), or pick another backend.")
        if shutil.which("ollama") is None:
            if shutil.which("zstd") is None:
                subprocess.run("apt-get -qq update && apt-get -qq install -y zstd", shell=True, check=True)
            subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)
        subprocess.Popen(["ollama", "serve"], stdout=open("ollama.log", "a"), stderr=subprocess.STDOUT)
        for _ in range(60):
            if ollama_running():
                break
            time.sleep(1)
        else:
            raise RuntimeError("Ollama did not start; see ollama.log.")
    if shutil.which("nvidia-smi") is None:
        print("No GPU found: Ollama will answer slowly on the CPU.")
    pulled = os.environ.get("LLM_MODEL", "llama3.2")
    if shutil.which("ollama"):
        subprocess.run(["ollama", "pull", pulled], check=True)

print(f"Ready: code in {root}, backend {BACKEND}" + (f", model {os.environ['LLM_MODEL']}" if os.environ.get("LLM_MODEL") else ""))

## Step 1: describe the tools

Import the lab's module and print what the model sees of each tool: a name, a description
and an input schema. The function behind each tool is never shown to it. See
[Step 1](https://handbook.aggarwal.dev/lab-agent.html#tools).

In [ ]:
sys.path.insert(0, str(root / "labs" / "05-agent"))
import json
import agent
from common import llm

for tool in agent.TOOLS:
    spec = tool.spec()
    print(f"{spec['name']:<18} side effect: {tool.side_effect!s:<5} | {spec['description'][:70]}...")

## Step 2: a calculator that cannot run code

Call `calculate` directly. The sum is exact, and anything that is not arithmetic comes back
as an error, however it is dressed up. See [Step 2](https://handbook.aggarwal.dev/lab-agent.html#calculator).

In [ ]:
for expression in ["2 * 6.49 + 18.50", "0.1 + 0.2", "2 x 6.49 + 18.50", "__import__('os').system('echo hi')", "1 / 0"]:
    try:
        print(f"{expression!r:<42} ->", agent.calculate(expression)["value"])
    except agent.ToolInputError as err:
        print(f"{expression!r:<42} -> error:", err)

## Step 3: check every request before it runs

`run_tool` turns every failure into an error result the model can read. Send it a request
with a bad quantity and one for a tool that does not exist. See
[Step 3](https://handbook.aggarwal.dev/lab-agent.html#check).

In [ ]:
bad_quantity = llm.ToolCall("t1", "place_order", {"items": [{"sku": "TUB-700-28", "quantity": 99}]})
unknown_tool = llm.ToolCall("t2", "weather", {"city": "York"})
for call in (bad_quantity, unknown_tool):
    out = agent.run_tool(call, approve=lambda c: False)
    print(out["is_error"], "|", out["content"])

## Step 4: a person approves side effects

The lab asks `y/N` on the keyboard. A notebook cannot wait for a keyboard, so these cells
pipe the answer in: `n` first, then `y`. Look for the error result after the refusal and the
order reference after the yes. See [Step 4](https://handbook.aggarwal.dev/lab-agent.html#approval).

In [ ]:
!printf 'n\n' | python labs/05-agent/agent.py --demo order --today 2026-11-02

In [ ]:
!printf 'y\n' | python labs/05-agent/agent.py --demo order --today 2026-11-02

## Step 5: the loop

Run the demo question: the model asks for tools, the code runs them and sends the results
back, until it answers. Watch the bad sum in step 2 come back as an error and be corrected,
and the input tokens climb with every call. See [Step 5](https://handbook.aggarwal.dev/lab-agent.html#loop).

In [ ]:
!python labs/05-agent/agent.py --today 2026-11-02

Now the step limit. With `--max-steps 2` the loop stops before running the second request.

In [ ]:
!python labs/05-agent/agent.py --max-steps 2 --today 2026-11-02

## Step 6: a stand-in model for offline runs

On the fake backend the lab plays the model with `pretend_model`, set the way the script
does it. Run the loop from Python and read the `Run` it returns. See
[Step 6](https://handbook.aggarwal.dev/lab-agent.html#fake).

In [ ]:
from datetime import date

if llm.backend_name() == "fake":
    llm.fake.responder = agent.pretend_model

run = agent.run_agent(agent.DEMOS["quote"], today=date(2026, 11, 2), say=lambda line: None)
print("stop:   ", run.stop)
print("answer: ", run.answer)
print("calls:  ", run.calls, "| tokens in/out:", run.input_tokens, "/", run.output_tokens)

## Run the tests

In [ ]:
!cd labs && python -m pytest -q -p no:cacheprovider 05-agent

## Experiments

Try the experiments on the [lab page](https://handbook.aggarwal.dev/lab-agent.html#experiments). To start the
first one, predict which request is left unrun with `--max-steps 3`, then run it here. The
fake backend costs nothing, and neither does Ollama.

In [ ]:
!python labs/05-agent/agent.py --max-steps 3 --today 2026-11-02

Then tick the lab's checklist in the book: your progress is kept in your browser, not in
this notebook.